# NB08 - Grokking: Delayed Generalization
## Background
Grokking (Power et al., 2022) is a striking training phenomenon: a network first overfits the training data (near-zero training loss, random test accuracy), then after thousands more steps, suddenly generalizes — test accuracy jumps from random to near-perfect. This 'delayed generalization' seems counterintuitive: why does more training after overfitting help?

The mechanism: weight norms continue growing during the overfit phase. Once weight decay brings the effective weight norm below a threshold, the network transitions to a qualitatively different solution that generalizes. Phase transitions, weight norm dynamics, and the connection to algorithmic tasks (modular arithmetic) make grokking a rich lens for understanding optimization.

## What You'll Learn
- Grokking demo on modular arithmetic (a+b mod p)
- Weight norm dynamics during the overfit and generalization phases
- How weight decay strength controls when (or whether) grokking occurs
- Effective representation analysis before and after the phase transition

## Why This Matters
Grokking revealed that training for longer than 'necessary' can improve generalization, contradicting early stopping intuitions. It provides insight into double descent, weight norm regularization, and why some models memorize vs. learn algorithms.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, List

# ── Modular arithmetic dataset ─────────────────────────────────────────────
def make_modular_dataset(p: int = 53, operation: str = 'add',
                          train_fraction: float = 0.4, seed: int = 42):
    """Create all (a, b, a op b mod p) triples. Train/test split."""
    np.random.seed(seed)
    pairs = [(a, b) for a in range(p) for b in range(p)]
    np.random.shuffle(pairs)
    X = np.array(pairs)
    if operation == 'add':
        y = (X[:, 0] + X[:, 1]) % p
    elif operation == 'mul':
        y = (X[:, 0] * X[:, 1]) % p
    n_train = int(len(X) * train_fraction)
    return X[:n_train], y[:n_train], X[n_train:], y[n_train:]

# ── Tiny MLP ─────────────────────────────────────────────────────────────
class TinyMLP:
    """2-layer MLP for modular arithmetic classification."""
    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int, seed: int = 0):
        np.random.seed(seed)
        scale = np.sqrt(2 / input_dim)
        self.W1 = np.random.randn(input_dim, hidden_dim) * scale
        self.b1 = np.zeros(hidden_dim)
        self.W2 = np.random.randn(hidden_dim, output_dim) * np.sqrt(2/hidden_dim)
        self.b2 = np.zeros(output_dim)

    def forward(self, x: np.ndarray) -> np.ndarray:
        self.h_pre = x @ self.W1 + self.b1
        self.h = np.maximum(0, self.h_pre)  # ReLU
        self.z = self.h @ self.W2 + self.b2
        # Softmax
        z_shifted = self.z - self.z.max(axis=-1, keepdims=True)
        exp_z = np.exp(z_shifted)
        self.probs = exp_z / exp_z.sum(axis=-1, keepdims=True)
        return self.probs

    def backward(self, x: np.ndarray, y: np.ndarray,
                  weight_decay: float = 0.0) -> dict:
        n = len(x)
        # Output layer gradient
        dz = self.probs.copy()
        dz[np.arange(n), y] -= 1
        dz /= n
        # W2, b2 gradients
        dW2 = self.h.T @ dz + weight_decay * self.W2
        db2 = dz.sum(axis=0)
        # Hidden layer
        dh = dz @ self.W2.T
        dh_pre = dh * (self.h_pre > 0)
        dW1 = x.T @ dh_pre + weight_decay * self.W1
        db1 = dh_pre.sum(axis=0)
        return {'W1': dW1, 'b1': db1, 'W2': dW2, 'b2': db2}

    def update(self, grads: dict, lr: float):
        for param_name, grad in grads.items():
            param = getattr(self, param_name)
            setattr(self, param_name, param - lr * grad)

    def weight_norm(self) -> float:
        return (np.linalg.norm(self.W1)**2 + np.linalg.norm(self.W2)**2)**0.5

    def cross_entropy_loss(self, probs: np.ndarray, y: np.ndarray) -> float:
        eps = 1e-12
        return -np.mean(np.log(probs[np.arange(len(y)), y] + eps))

    def accuracy(self, x: np.ndarray, y: np.ndarray) -> float:
        probs = self.forward(x)
        return np.mean(np.argmax(probs, axis=-1) == y)

print('TinyMLP and modular arithmetic dataset ready.')

In [ ]:
# ── One-hot encode integer inputs ─────────────────────────────────────────
p = 53
X_tr, y_tr, X_te, y_te = make_modular_dataset(p, 'add', train_fraction=0.4)

def one_hot(arr, n):
    oh = np.zeros((len(arr), n), dtype=float)
    oh[np.arange(len(arr)), arr] = 1.0
    return oh

X_tr_enc = np.concatenate([one_hot(X_tr[:, 0], p), one_hot(X_tr[:, 1], p)], axis=1)
X_te_enc = np.concatenate([one_hot(X_te[:, 0], p), one_hot(X_te[:, 1], p)], axis=1)

# ── Train with and without weight decay ───────────────────────────────────
def train_model(weight_decay: float, n_steps: int = 20000, lr: float = 0.01,
                hidden_dim: int = 128, seed: int = 42):
    model = TinyMLP(2*p, hidden_dim, p, seed=seed)
    train_accs, test_accs, weight_norms = [], [], []

    for step in range(n_steps):
        probs = model.forward(X_tr_enc)
        grads = model.backward(X_tr_enc, y_tr, weight_decay=weight_decay)
        model.update(grads, lr=lr)

        if step % 100 == 0:
            train_accs.append(model.accuracy(X_tr_enc, y_tr))
            test_accs.append(model.accuracy(X_te_enc, y_te))
            weight_norms.append(model.weight_norm())

    return train_accs, test_accs, weight_norms

print('Training with weight_decay=0.01 (expect grokking)...')
tr1, te1, wn1 = train_model(weight_decay=0.01, n_steps=15000)
print(f'  Final train acc: {tr1[-1]:.3f}, test acc: {te1[-1]:.3f}')

print('Training with weight_decay=0.0 (no regularization)...')
tr0, te0, wn0 = train_model(weight_decay=0.0, n_steps=15000)
print(f'  Final train acc: {tr0[-1]:.3f}, test acc: {te0[-1]:.3f}')

In [ ]:
steps_x = np.arange(len(tr1)) * 100

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (tr, te, wn, label) in zip(axes, [
    (tr1, te1, wn1, 'wd=0.01 (grokking)'),
    (tr0, te0, wn0, 'wd=0.0 (no grokking)'),
]):
    ax.plot(steps_x, tr, label='Train acc', linewidth=1.5)
    ax.plot(steps_x, te, label='Test acc', linewidth=1.5)
    ax.set_title(f'Accuracy: {label}')
    ax.set_xlabel('Training steps'); ax.set_ylabel('Accuracy')
    ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(0, 1.05)

axes[2].plot(steps_x, wn1, label='wd=0.01', linewidth=1.5)
axes[2].plot(steps_x, wn0, label='wd=0.0', linewidth=1.5)
axes[2].set_title('Weight Norm Dynamics')
axes[2].set_xlabel('Training steps'); axes[2].set_ylabel('||W||')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{BASE}/s30_08_grokking.png', dpi=100, bbox_inches='tight')
plt.show()
print()
print('Grokking interpretation:')
print('  Phase 1 (memorization): network memorizes training examples')
print('  Phase 2 (circuit formation): weight decay shrinks inefficient weights')
print('  Phase 3 (generalization): a compact algorithmic circuit emerges')
print('  Weight decay is essential - without it, no grokking!')